In [7]:
import os
from tqdm import tqdm
import sys
if not hasattr(sys.modules[__name__], "cwd_changed"):
    os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))
    sys.modules[__name__].cwd_changed = True

import warnings 
warnings.filterwarnings("ignore")
import pandas as pd
from utils.data import read_data_files

# Exploring triplet recovery

## Interaction Information

In [8]:
from metrics.synergy import find_synergistic_triplets
from utils.data import read_data_files
import os
import ast
from tqdm import tqdm


def triplets_found(df, metadata_df, top_n=10):
    # triplets, scores = zip(*find_synergistic_triplets(df, inflection_point=True))
    triplets = find_synergistic_triplets(df, inflection_point=True)
    triplets_dict = {}  # Dictionary to store results
    
    if "SRV" in metadata_df["Type"].values:
        all_colliders = metadata_df["Combs"] 
        synergistic_colliders = metadata_df[metadata_df["Type"] == "SRV"]["Combs"]
    else:
        all_colliders = metadata_df["Combs"]
        synergistic_colliders = metadata_df[metadata_df["Type"] == "XOR"]["Combs"]

    found = 0
    triplets_found = []
    for check in synergistic_colliders:
        # print("Checking triplet:", check)
        check_list = ast.literal_eval(check)
        for i in triplets[:top_n]:
            candidate_triplet = [int(j) for j in i]
            # print("Candidate triplet:", candidate_triplet)
            if candidate_triplet == check_list:
                # print(f"Found triplet {check} in results.")
                triplets_found.append(check_list)
                found += 1
    # print(f"Total synergistic triplets found: {found} out of {len(synergistic_colliders)}")
    triplets_ratio_found = found / len(synergistic_colliders) if len(synergistic_colliders) > 0 else 0
    
    triplets_dict["Ratio"] = triplets_ratio_found
    triplets_dict["Total"] = len(synergistic_colliders)
    triplets_dict["Found"] = found
    triplets_dict["N_Vars"] = df.shape[1]
    triplets_dict["Metric"] = "InteractionInformation"
    triplets_dict["Directory"] = directory.split("\\")[-1]
    return triplets_dict, triplets

In [9]:
import pickle


def get_triplet_results(directory, csv_by_id, metadata_by_id):
    triplets_dict_list = []
    results_path = f"data/triplets/{directory.split(os.sep)[-1]}.pkl"
    # Try to load intermediate results if they exist
    if os.path.isfile(results_path):
        with open(results_path, "rb") as f:
            triplets_dict_list = pickle.load(f)
        processed_files = {d["File"] for d in triplets_dict_list if "File" in d}
    else:
        print(f"Could not load intermediate results from {results_path}. Starting fresh.")
        triplets_dict_list = []
        processed_files = set()


    for i in tqdm(list(csv_by_id.keys()), desc=f"Processing files in {directory}"):
        if i in processed_files:
            continue
        df = csv_by_id[i]
        metadata_df = metadata_by_id[i]
        triplets_dict, triplets = triplets_found(df, metadata_df)
        triplets_dict["File"] = i
        triplets_dict["Directory"] = directory.split(os.sep)[-1]
        triplets_dict["Triplets"] = triplets
        triplets_dict_list.append(triplets_dict)
        # Save intermediate results after every 20 files or if file does not exist
        if len(triplets_dict_list) % 20 == 0:
            with open(results_path, "wb") as f:
                pickle.dump(triplets_dict_list, f)

    # Final Save File
    with open(results_path, "wb") as f:
        pickle.dump(triplets_dict_list, f)

    results_df = pd.DataFrame(triplets_dict_list)

    return results_df

In [10]:
def save_triplet_results(directory):
    csv_by_id, metadata_by_id, = read_data_files(directory)
    results_df = get_triplet_results(directory, csv_by_id, metadata_by_id)
    # print(f"Saved results to {os.path.join(child, 'triplet_synergy_results.csv')}")
    results_df.to_csv(f"results/{directory.split(os.sep)[-1]}_triplets.csv", index=False)

In [11]:
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=RuntimeWarning)

directory  = "data/datasets/binary_data"
# get_triplet_results(directory, csv_by_id, metadata_by_id)
# save_triplet_results(directory)   

In [12]:
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=RuntimeWarning)

directory  = "data/datasets/jpmf_data"
# get_triplet_results(directory, csv_by_id, metadata_by_id)
save_triplet_results(directory)

Processing files in data/datasets/jpmf_data: 100%|██████████| 720/720 [11:09<00:00,  1.08it/s] 
